# Final Analysis — Multi-Seed Replication

**No training happens in this notebook.** It reads the artifacts produced by `Experiment1_MultiSeed.ipynb` and `Experiment3_MultiSeed.ipynb` and produces publication-ready tables, figures, and significance tests.

**Claims under test**
- **Claim 1:** MNRL consistently outperforms the pretrained MiniLM baseline.
- **Claim 2:** MNRL consistently outperforms TripletLoss under identical conditions.

**Statistical design — two complementary levels of evidence:**
1. **Across seeds (n = 5):** the unit of analysis is a training run. Claim 1 uses a one-sample t-test against the fixed, deterministic baseline constant; Claim 2 uses a **paired** t-test and Wilcoxon signed-rank test on seed-matched runs. At n = 5 these have low power (Wilcoxon's minimum attainable one-sided p is 0.03125), so effect sizes and per-seed win counts are reported alongside p-values.
2. **Across queries (n = 431 test queries):** the unit is a query. Saved per-query retrieval indices give bootstrap confidence intervals on the mean metric difference and McNemar's exact test on paired per-query Hit@10. This is the higher-powered evidence.

Both levels are reported because neither alone is sufficient: across-seed tests establish that the effect is not a seed artifact; across-query tests establish that it is not sampling noise over queries.

Upload `exp1_multiseed.zip` and `exp3_multiseed.zip` when prompted.

In [ ]:
%pip install -q pandas numpy scipy matplotlib tabulate
import os, zipfile, glob, json, re, shutil
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

METRICS = ["HitRate@10", "Recall@10", "MRR@10", "nDCG@10",
           "HitRate@25", "Recall@25", "MRR@25", "nDCG@25"]
# internal keys in baseline_metrics.json  <->  public column names
KEYMAP = {"HitRate": "hit_rate", "Recall": "recall", "MRR": "mrr", "nDCG": "ndcg"}
print("ready")

In [ ]:
# --- Upload both bundles; extract each into its OWN directory ---
# Separate directories matter: each bundle carries its own baseline artifacts,
# and extracting both into one tree would let one silently overwrite the other.
# Keeping them apart lets the next cell verify they agree.
os.makedirs("bundle", exist_ok=True)
if not (os.path.exists("bundle/exp1") and os.path.exists("bundle/exp3")):
    from google.colab import files
    up = files.upload()   # select exp1_multiseed.zip AND exp3_multiseed.zip
    for name in up:
        tmp = f"bundle/_tmp_{name}"
        with zipfile.ZipFile(name) as zf:
            zf.extractall(tmp)
        which = "exp1" if glob.glob(f"{tmp}/outputs/results/exp1_seed_results.csv") else "exp3"
        shutil.rmtree(f"bundle/{which}", ignore_errors=True)
        os.rename(tmp, f"bundle/{which}")
        print(f"{name} -> bundle/{which}")

P1, P3 = "bundle/exp1/outputs/results", "bundle/exp3/outputs/results"
assert os.path.exists(f"{P1}/exp1_seed_results.csv"), "exp1 bundle missing"
assert os.path.exists(f"{P3}/exp3_seed_results.csv"), "exp3 bundle missing"
print("both bundles present")

In [ ]:
# --- INTEGRITY CHECK: the two runs must share an identical evaluation basis ---
# Known issue this guards against: retrieval backends can break score ties
# differently, which leaves set-based metrics (HitRate/Recall) identical while
# shifting rank-based ones (MRR/nDCG) by ~0.002. That is far below the effect
# under study, but it must be detected and reported, not silently absorbed.
b1 = json.load(open(f"{P1}/baseline_metrics.json"))
b3 = json.load(open(f"{P3}/baseline_metrics.json"))
keys = [f"{KEYMAP[m.split('@')[0]]}@{m.split('@')[1]}" for m in METRICS]
diffs = {k: abs(b1[k] - b3[k]) for k in keys}
maxdiff = max(diffs.values())
print(f"Baseline agreement between bundles: max |diff| = {maxdiff:.2e}")
for k, d in diffs.items():
    if d > 1e-9:
        print(f"   {k}: exp1={b1[k]:.4f}  exp3={b3[k]:.4f}  diff={d:+.4f}")
if maxdiff < 1e-9:
    print("OK — identical baseline; the two experiments share one evaluation basis.")
else:
    print("WARNING — baselines differ between bundles. Report this in the paper.")
    print("Comparisons below use the exp3 bundle's baseline for BOTH experiments,")
    print("so Claim 1 and Claim 2 are evaluated against one consistent reference.")

# splits must also match, or per-query comparisons are not aligned
for f in ["test_qdp.csv", "train_qdp.csv", "val_qdp.csv"]:
    same = pd.read_csv(f"{P1}/{f}").equals(pd.read_csv(f"{P3}/{f}"))
    print(f"{f}: identical across bundles = {same}")
    assert same, f"{f} differs between bundles — comparison would be invalid"

BASELINE_SRC = P3  # single canonical reference for all comparisons
baseline = json.load(open(f"{BASELINE_SRC}/baseline_metrics.json"))
BASE = {m: baseline[f"{KEYMAP[m.split('@')[0]]}@{m.split('@')[1]}"] for m in METRICS}

In [ ]:
# --- Load per-seed results ---
e1 = pd.read_csv(f"{P1}/exp1_seed_results.csv").sort_values("seed").reset_index(drop=True)
e3 = pd.read_csv(f"{P3}/exp3_seed_results.csv").sort_values("seed").reset_index(drop=True)
assert list(e1.seed) == list(e3.seed), "seed sets differ — pairing would be invalid"
print("seeds:", list(e1.seed))
print("\nselected checkpoints per seed:")
print(pd.DataFrame({"seed": e1.seed, "Exp1": e1.selected_checkpoint, "Exp3": e3.selected_checkpoint}).to_string(index=False))
print("\nbaseline (deterministic, never retrained):")
print(pd.Series(BASE).round(4).to_string())

In [ ]:
# --- TABLE 1: Baseline vs Experiment 3 (mean ± std over seeds), test set ---
rows = []
for m in METRICS:
    v = e3[f"test_{m}"].values
    rows.append({"Metric": m,
                 "Baseline": f"{BASE[m]:.4f}",
                 "MNRL (mean ± s.d.)": f"{v.mean():.4f} ± {v.std(ddof=1):.4f}",
                 "Δ abs": f"{v.mean()-BASE[m]:+.4f}",
                 "Δ %": f"{(v.mean()-BASE[m])/BASE[m]*100:+.2f}%",
                 "seeds > baseline": f"{int((v > BASE[m]).sum())}/{len(v)}"})
table1 = pd.DataFrame(rows)
os.makedirs("analysis", exist_ok=True)
table1.to_csv("analysis/table1_baseline_vs_exp3.csv", index=False)
print("TABLE 1 — Pretrained baseline vs Experiment 3 (MNRL), held-out test set")
table1

In [ ]:
# --- TABLE 2: Experiment 1 vs Experiment 3 (mean ± std over seeds), test set ---
rows = []
for m in METRICS:
    a, b = e1[f"test_{m}"].values, e3[f"test_{m}"].values
    rows.append({"Metric": m,
                 "TripletLoss (mean ± s.d.)": f"{a.mean():.4f} ± {a.std(ddof=1):.4f}",
                 "MNRL (mean ± s.d.)": f"{b.mean():.4f} ± {b.std(ddof=1):.4f}",
                 "Δ abs": f"{b.mean()-a.mean():+.4f}",
                 "Δ %": f"{(b.mean()-a.mean())/a.mean()*100:+.2f}%",
                 "seeds MNRL > Triplet": f"{int((b > a).sum())}/{len(a)}"})
table2 = pd.DataFrame(rows)
table2.to_csv("analysis/table2_exp1_vs_exp3.csv", index=False)
print("TABLE 2 — Experiment 1 (TripletLoss) vs Experiment 3 (MNRL), held-out test set")
table2

In [ ]:
# --- Significance ACROSS SEEDS (n = 5 runs) ---
# Claim 1: one-sample t-test vs the baseline CONSTANT (baseline is deterministic,
#          so it is not a sample and a paired test against it is not defined).
# Claim 2: paired t-test + Wilcoxon signed-rank on seed-matched runs.
rows = []
for m in METRICS:
    a, b = e1[f"test_{m}"].values, e3[f"test_{m}"].values
    sb = b.std(ddof=1)
    t1, p1 = stats.ttest_1samp(b, BASE[m], alternative="greater")
    d1 = (b.mean() - BASE[m]) / sb if sb > 0 else np.inf
    t2, p2 = stats.ttest_rel(b, a, alternative="greater")
    diff = b - a
    sd = diff.std(ddof=1)
    d2 = diff.mean() / sd if sd > 0 else np.inf
    try:
        _, pw = stats.wilcoxon(b, a, alternative="greater")
    except ValueError:
        pw = np.nan
    rows.append({"Metric": m,
                 "C1 t": f"{t1:.2f}", "C1 p": f"{p1:.4g}", "C1 d": f"{d1:.2f}",
                 "C2 t(paired)": f"{t2:.2f}", "C2 p": f"{p2:.4g}",
                 "C2 Wilcoxon p": (f"{pw:.4g}" if pw == pw else "n/a"),
                 "C2 d": f"{d2:.2f}"})
seed_tests = pd.DataFrame(rows)
seed_tests.to_csv("analysis/significance_across_seeds.csv", index=False)
print("Across-seed significance (one-sided). C1 = MNRL > baseline; C2 = MNRL > TripletLoss.")
print(f"n = {len(e3)} seeds. Wilcoxon's minimum attainable one-sided p at n=5 is 0.03125.")
seed_tests

In [ ]:
# --- Per-query machinery: rebuild per-query metrics from saved retrieval indices ---
corpus = pd.concat([pd.read_csv(f"{BASELINE_SRC}/train_qdp.csv"),
                    pd.read_csv(f"{BASELINE_SRC}/val_qdp.csv")], ignore_index=True)
queries = pd.read_csv(f"{BASELINE_SRC}/test_qdp.csv")
CM, QM = corpus["MisconceptionId"].values, queries["MisconceptionId"].values

def per_query(indices, k=10):
    """Per-query (hit@k, reciprocal_rank@k); relevance = shared MisconceptionId."""
    rel = (CM[indices[:, :k]] == QM[:, None])
    hit = rel.any(axis=1).astype(float)
    rr = np.zeros(len(rel))
    for i, r in enumerate(rel):
        w = np.where(r)[0]
        rr[i] = 1.0 / (w[0] + 1) if len(w) else 0.0
    return hit, rr

base_hit, base_rr = per_query(np.load(f"{BASELINE_SRC}/baseline_indices.npy"))
gap_hit = abs(base_hit.mean() - BASE["HitRate@10"])
gap_mrr = abs(base_rr.mean() - BASE["MRR@10"])
print(f"queries: {len(QM)}")
print(f"recomputed baseline  Hit@10={base_hit.mean():.4f}  MRR@10={base_rr.mean():.4f}")
print(f"reported  baseline   Hit@10={BASE['HitRate@10']:.4f}  MRR@10={BASE['MRR@10']:.4f}")

# The saved indices and saved metrics must describe the same evaluation.
# Tolerance 1e-3 is far below the effect under study (~0.04) but tight enough
# to catch a genuine mismatch (e.g. indices and metrics from different runs).
PERQUERY_OK = (gap_hit < 1e-3) and (gap_mrr < 1e-3)
if PERQUERY_OK:
    print("OK — per-query reconstruction agrees with the saved metrics.")
else:
    print("\n*** WARNING: per-query reconstruction disagrees with saved metrics ***")
    print(f"    Hit@10 gap = {gap_hit:.5f}, MRR@10 gap = {gap_mrr:.5f}")
    print("    If ONLY the rank-based gap is non-zero, the cause is tie-breaking")
    print("    between retrieval backends: identical retrieved sets, different")
    print("    within-set ordering. If the Hit@10 gap is also non-zero, the")
    print("    indices and metrics come from different runs and the per-query")
    print("    tests below would be invalid.")
    print("    Per-query results are still computed, but treat them as")
    print("    indicative and report the discrepancy.")

In [ ]:
# --- Per-query significance: bootstrap CIs + McNemar, per seed ---
def bootstrap_ci(diff, n_boot=10000, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.randint(0, len(diff), size=(n_boot, len(diff)))
    means = diff[idx].mean(axis=1)
    return np.percentile(means, 2.5), np.percentile(means, 97.5)

def mcnemar(h_a, h_b):
    """One-sided exact McNemar: is b better than a on paired binary outcomes?"""
    bb = int(((h_a == 1) & (h_b == 0)).sum())
    cc = int(((h_a == 0) & (h_b == 1)).sum())
    p = stats.binomtest(cc, bb + cc, 0.5, alternative="greater").pvalue if bb + cc else 1.0
    return bb, cc, p

rows, pooled = [], {"e1_rr": [], "e3_rr": [], "e1_hit": [], "e3_hit": []}
for seed in e3.seed:
    f1 = f"{P1}/exp1_seed{seed}_selected_indices.npy"
    f3 = f"{P3}/exp3_seed{seed}_selected_indices.npy"
    if not (os.path.exists(f1) and os.path.exists(f3)):
        print(f"seed {seed}: per-query indices missing — skipped"); continue
    h1, r1 = per_query(np.load(f1)); h3, r3 = per_query(np.load(f3))
    pooled["e1_rr"].append(r1); pooled["e3_rr"].append(r3)
    pooled["e1_hit"].append(h1); pooled["e3_hit"].append(h3)
    lo_b, hi_b = bootstrap_ci(r3 - base_rr); _, _, p_b = mcnemar(base_hit, h3)
    lo_t, hi_t = bootstrap_ci(r3 - r1);      _, _, p_t = mcnemar(h1, h3)
    rows.append({"seed": seed,
                 "ΔMRR@10 vs base": f"{(r3-base_rr).mean():+.4f}",
                 "95% CI": f"[{lo_b:+.4f}, {hi_b:+.4f}]",
                 "McNemar p": f"{p_b:.3g}",
                 "ΔMRR@10 vs Triplet": f"{(r3-r1).mean():+.4f}",
                 "95% CI ": f"[{lo_t:+.4f}, {hi_t:+.4f}]",
                 "McNemar p ": f"{p_t:.3g}"})
perq = pd.DataFrame(rows)
perq.to_csv("analysis/significance_per_query.csv", index=False)
print(f"Per-query significance (n = {len(QM)} queries; CIs bootstrapped over queries)")
perq

In [ ]:
# --- Pooled per-query evidence (per-query scores averaged across seeds) ---
if pooled["e3_rr"]:
    e3m, e1m = np.mean(pooled["e3_rr"], axis=0), np.mean(pooled["e1_rr"], axis=0)
    lo1, hi1 = bootstrap_ci(e3m - base_rr)
    lo2, hi2 = bootstrap_ci(e3m - e1m)
    print("POOLED per-query bootstrap (seed-averaged scores, resampled over queries)")
    print(f"  Claim 1  MNRL - baseline: ΔMRR@10 = {(e3m-base_rr).mean():+.4f}  95% CI [{lo1:+.4f}, {hi1:+.4f}]")
    print(f"  Claim 2  MNRL - Triplet : ΔMRR@10 = {(e3m-e1m).mean():+.4f}  95% CI [{lo2:+.4f}, {hi2:+.4f}]")
    print("  A CI excluding 0 supports the claim.")
    json.dump({"claim1_delta_mrr10": float((e3m-base_rr).mean()), "claim1_ci95": [float(lo1), float(hi1)],
               "claim2_delta_mrr10": float((e3m-e1m).mean()), "claim2_ci95": [float(lo2), float(hi2)]},
              open("analysis/pooled_bootstrap.json", "w"), indent=2)
else:
    print("No per-query indices found. To enable per-query tests, ensure the")
    print("'{exp}_seed{S}_selected_indices.npy' files are included in each bundle.")

In [ ]:
# --- FIGURE 1: mean ± s.d. across seeds, with baseline reference ---
plt.rcParams.update({"font.family": "serif", "font.size": 11, "savefig.dpi": 300,
                     "axes.grid": True, "grid.alpha": 0.3})
os.makedirs("analysis/figures", exist_ok=True)
show = ["HitRate@10", "Recall@10", "MRR@10", "nDCG@10"]
x, w = np.arange(len(show)), 0.35
fig, ax = plt.subplots(figsize=(9, 5))
m1 = [e1[f"test_{m}"].mean() for m in show]; s1 = [e1[f"test_{m}"].std(ddof=1) for m in show]
m3 = [e3[f"test_{m}"].mean() for m in show]; s3 = [e3[f"test_{m}"].std(ddof=1) for m in show]
ax.bar(x - w/2, m1, w, yerr=s1, capsize=4, label="TripletLoss (Exp 1)", color="#94a3b8", edgecolor="white")
ax.bar(x + w/2, m3, w, yerr=s3, capsize=4, label="MNRL (Exp 3)", color="#2563eb", edgecolor="white")
for i, m in enumerate(show):
    ax.hlines(BASE[m], i - 0.5, i + 0.5, color="#0f172a", ls="--", lw=1.5,
              label="pretrained baseline" if i == 0 else None)
ax.set_xticks(x); ax.set_xticklabels(show); ax.set_ylabel("test metric value")
ax.set_title(f"Retrieval on held-out test set (mean ± s.d. over {len(e3)} seeds)")
ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig("analysis/figures/fig1_bars.png", bbox_inches="tight")
plt.show()

In [ ]:
# --- FIGURE 2: per-seed distributions (box + individual runs) ---
fig, axes = plt.subplots(1, len(show), figsize=(4*len(show), 4.2))
jit = np.random.RandomState(0).uniform(-0.07, 0.07, len(e3))
for ax, m in zip(axes, show):
    d1, d3 = e1[f"test_{m}"].values, e3[f"test_{m}"].values
    bp = ax.boxplot([d1, d3], widths=0.5, patch_artist=True, tick_labels=["Triplet", "MNRL"])
    for patch, c in zip(bp["boxes"], ["#94a3b8", "#2563eb"]):
        patch.set_facecolor(c); patch.set_alpha(0.55)
    for j, d in enumerate([d1, d3], start=1):
        ax.scatter(np.full(len(d), j) + jit, d, color="#0f172a", s=18, zorder=3)
    ax.axhline(BASE[m], color="#dc2626", ls="--", lw=1.3)
    ax.set_title(m); ax.set_ylabel("test value")
fig.suptitle("Per-seed distributions (red dashed = pretrained baseline)", y=1.02)
fig.tight_layout(); fig.savefig("analysis/figures/fig2_box.png", bbox_inches="tight")
plt.show()

In [ ]:
# --- Verdict + package ---
n = len(e3)
c1 = int((e3["test_MRR@10"] > BASE["MRR@10"]).sum())
c2 = int((e3["test_MRR@10"].values > e1["test_MRR@10"].values).sum())
print(f"CLAIM 1  MNRL > baseline    : {c1}/{n} seeds | "
      f"mean MRR@10 {e3['test_MRR@10'].mean():.4f} vs {BASE['MRR@10']:.4f}")
print(f"CLAIM 2  MNRL > TripletLoss : {c2}/{n} seeds | "
      f"mean MRR@10 {e3['test_MRR@10'].mean():.4f} vs {e1['test_MRR@10'].mean():.4f}")
print("\nA claim is well supported when all three hold:")
print("  (a) win count is unanimous or near-unanimous across seeds,")
print("  (b) the across-seed one-sided p < 0.05,")
print("  (c) the per-query bootstrap 95% CI excludes 0.")

for src, dst in [(f"{P1}/exp1_summary.csv", "analysis/exp1_summary.csv"),
                 (f"{P3}/exp3_summary.csv", "analysis/exp3_summary.csv"),
                 (f"{P1}/exp1_seed_results.csv", "analysis/exp1_seed_results.csv"),
                 (f"{P3}/exp3_seed_results.csv", "analysis/exp3_seed_results.csv")]:
    if os.path.exists(src): shutil.copy(src, dst)

!zip -qr final_analysis.zip analysis
from google.colab import files as colab_files
colab_files.download("final_analysis.zip")